# Antithetic DiffPDHG — FFHQ-100 Single Run in Colab

This notebook runs one configurable DiffPDHG experiment and patches the cloned repository locally to support

\[
x^{k+1}=\frac12\left[D_{\sigma_k}(v^{k+1}+a_kn^k)+D_{\sigma_k}(v^{k+1}-a_kn^k)\right].
\]

The default uses `a_k = sigma_k` during the theorem-style tail. Each score input is queried at the training noise scale, while the explicit `+/- sigma_k n^k` displacement cancels in the averaged primal output.

The patch is applied only inside the Colab checkout; it does not modify the GitHub repository.


## Runtime

In Colab, go to `Runtime > Change runtime type` and choose:
- `Python 3`
- `GPU`

Prefer `A100` or `H100`. If `batch_size=100` is too large for the runtime, reduce it to `50` or `25` and rerun.

In [ ]:
#@title Project and Antithetic Settings

SETUP_MODE = "git"  #@param ["git", "drive_zip"]
REPO_URL = "https://github.com/Seif-Hussein/dyscode.git"  #@param {type:"string"}
REPO_BRANCH = "codex-pdhg-colab-light-100"  #@param {type:"string"}
DRIVE_ZIP_PATH = "/content/drive/MyDrive/mycode2.zip"  #@param {type:"string"}

REPO_DIR = "/content/mycode2"  #@param {type:"string"}
PYTHON_BIN = "/usr/bin/python3"  #@param {type:"string"}
DRIVE_EXPORT_DIR = "/content/drive/MyDrive/pdhg_single_run_exports"  #@param {type:"string"}
DRIVE_FFHQ_DATA_DIR = "/content/drive/MyDrive/mycode/ffhq256"  #@param {type:"string"}
SESSION_TAG = ""  #@param {type:"string"}
RUN_NAME = "Antithetic_DiffPDHG_FFHQ256"  #@param {type:"string"}
CONFIG_NAME = "default_ffhq.yaml"  #@param {type:"string"}

SAMPLER_CONFIG = "edm_pdhg"  #@param ["edm_pdhg", "edm_admm"]
INVERSE_TASK = "phase_retrieval"  #@param ["phase_retrieval", "phase_retrieval_explicit", "inpainting", "inpainting_explicit", "inpainting_rand", "inpainting_rand_explicit", "motion_blur", "motion_blur_explicit", "gaussian_blur", "gaussian_blur_explicit", "down_sampling", "down_sampling_explicit", "hdr", "hdr_explicit"]

SEED = 99  #@param {type:"integer"}
TOTAL_IMAGES = 100  #@param {type:"integer"}
BATCH_SIZE = 100  #@param {type:"integer"}
DATA_START_IDX = 0  #@param {type:"integer"}

NUM_STEPS = 500  #@param {type:"integer"}
MAX_ITER = 500  #@param {type:"integer"}
EARLY_STOP_ITER = 0  #@param {type:"integer"}
SIGMA_MAX = 27.0  #@param {type:"number"}
SIGMA_MIN = 0.075  #@param {type:"number"}
TAU = 0.01  #@param {type:"number"}
SIGMA_DUAL = 1600.0  #@param {type:"number"}
SIGMA_DUAL_SCHEDULE_MODE = "to_infinity"  #@param ["constant", "to_zero", "to_infinity"]
SIGMA_DUAL_SCHEDULE_SCOPE = "tail"  #@param ["full", "tail"]
SIGMA_DUAL_SCHEDULE_POWER = 2.5  #@param {type:"number"}
RHO = 500.0  #@param {type:"number"}
ADMM_LGVD_NUM_STEPS = 10  #@param {type:"integer"}
ADMM_ML_LR = 0.1  #@param {type:"number"}
DENOISER_AC_NOISE = True  #@param {type:"boolean"}
MEASUREMENT_SIGMA = 0.05  #@param {type:"number"}
DENOISE_FINAL_STEP = "tweedie"  #@param ["tweedie", "ode"]

# Antithetic mode is implemented for PDHG + direct Tweedie denoising (zero inner LGVD steps).
ANTITHETIC_ENABLED = True  #@param {type:"boolean"}
ANTITHETIC_SCOPE = "theorem1_tail"  #@param ["full", "last_n", "theorem1_tail"]
ANTITHETIC_LAST_N = 50  #@param {type:"integer"}
ANTITHETIC_NOISE_SOURCE = "sigma"  #@param ["sigma", "rho"]
ANTITHETIC_EXECUTION = "sequential"  #@param ["sequential", "stacked"]

# Optional asymptotic tail. The tail uses sigma_k^2 ~ 1/k and tau_k proportional to sigma_k^2.
# For ANTITHETIC_SCOPE="theorem1_tail", set this to a positive value.
MAP_TAIL_STEPS = 50  #@param {type:"integer"}
MAP_TAIL_SIGMA_MODE = "theorem1"  #@param ["theorem1", "linear"]
MAP_TAIL_RHO_POWER = 2.1  #@param {type:"number"}
MAP_TAIL_RHO_SCALE = 1.0  #@param {type:"number"}
MAP_TAIL_LAMBDA = 0.0  #@param {type:"number"}  # 0 means infer lambda from continuity at the switch

DYS_GAMMA = 0.0075  #@param {type:"number"}
LAMBDA_START = 1.0  #@param {type:"number"}
LAMBDA_END = 1.0  #@param {type:"number"}
LAMBDA_WARMUP = 0  #@param {type:"integer"}

EVAL_METRICS = "psnr;ssim;lpips"  #@param {type:"string"}
SAVE_SAMPLES = False  #@param {type:"boolean"}
SAVE_TRAJ = False  #@param {type:"boolean"}
SAVE_TRAJ_RAW_DATA = False  #@param {type:"boolean"}
LOG_TAIL_LINES = 120  #@param {type:"integer"}

# Optional extra Hydra overrides, separated by semicolons.
# Example for box inpainting: inverse_task.operator.mask_type=box;inverse_task.operator.mask_len_range=[128,129]
# Example for random inpainting: inverse_task.operator.mask_type=random;inverse_task.operator.mask_prob_range=[0.70,0.71]
EXTRA_HYDRA_OVERRIDES = ""  #@param {type:"string"}


In [ ]:
#@title Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#@title Fetch The Repo
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

repo_dir = Path(REPO_DIR)
repo_dir.parent.mkdir(parents=True, exist_ok=True)
os.chdir(repo_dir.parent)

if repo_dir.exists():
    shutil.rmtree(repo_dir)

if SETUP_MODE == "git":
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            REPO_BRANCH,
            "--single-branch",
            REPO_URL,
            repo_dir.as_posix(),
        ],
        check=True,
    )
elif SETUP_MODE == "drive_zip":
    zip_path = Path(DRIVE_ZIP_PATH)
    if not zip_path.exists():
        raise FileNotFoundError(f"Zip file not found: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(repo_dir.parent)
    extracted_root = repo_dir.parent / zip_path.stem
    if extracted_root.exists() and extracted_root != repo_dir:
        if repo_dir.exists():
            shutil.rmtree(repo_dir)
        extracted_root.rename(repo_dir)
else:
    raise ValueError(f"Unsupported SETUP_MODE: {SETUP_MODE}")

os.chdir(repo_dir)
print(f"Repo ready: {repo_dir}")


In [ ]:
#@title Apply the Antithetic DiffPDHG Patch
from pathlib import Path
import py_compile

pdhg_path = Path(REPO_DIR) / "sampler" / "pdhg.py"
if not pdhg_path.exists():
    raise FileNotFoundError(f"PDHG implementation not found: {pdhg_path}")

source = pdhg_path.read_text(encoding="utf-8")
marker = "ANTITHETIC_DIFFPDHG_PATCH_V1"

if marker in source:
    print(f"Antithetic patch already present in {pdhg_path}")
else:
    old_signature = (
        '    def optimize_denoising(self, z_in, model, d_k, sigma, rho=None, '
        'prior_use_type="denoise", wandb=False):'
    )
    new_signature = (
        '    def optimize_denoising(self, z_in, model, d_k, sigma, rho=None, '
        'prior_use_type="denoise", wandb=False, antithetic=False, '
        'antithetic_execution="sequential"):'
    )
    if old_signature not in source:
        raise RuntimeError("Could not find the expected optimize_denoising signature. The branch may have changed.")
    source = source.replace(old_signature, new_signature, 1)

    old_denoise_anchor = '''            if prior_use_type not in ["denoise"]:
                raise Exception(f"Prior type {prior_use_type} not supported!!!")

            ac_noise = bool(getattr(denoise_config, "ac_noise", True))
'''
    new_denoise_anchor = '''            if prior_use_type not in ["denoise"]:
                raise Exception(f"Prior type {prior_use_type} not supported!!!")

            # ANTITHETIC_DIFFPDHG_PATCH_V1
            # Every marginal query is at z_in +/- rho*n with the same conditioning sigma.
            # Averaging the two Tweedie outputs cancels the explicit +/-rho*n term.
            if bool(antithetic):
                if denoise_config.final_step != "tweedie":
                    raise NotImplementedError(
                        "Antithetic DiffPDHG currently requires denoise.final_step='tweedie'."
                    )
                num_steps = int(getattr(denoise_config.lgvd, "num_steps", 0))
                if num_steps != 0:
                    raise NotImplementedError(
                        "Theory-aligned antithetic mode requires denoise.lgvd.num_steps=0. "
                        "Otherwise the two inner stochastic trajectories must also be antithetically coupled."
                    )

                pair_noise = torch.randn_like(noisy_im)
                pair_perturb = pair_noise * rho_view
                forward_plus = noisy_im + pair_perturb
                forward_minus = noisy_im - pair_perturb

                execution = str(antithetic_execution).lower()
                if execution == "stacked":
                    pair_input = torch.cat([forward_plus, forward_minus], dim=0)
                    pair_sigma = torch.cat([sigma_batch, sigma_batch], dim=0)
                    pair_output = model.tweedie(pair_input, pair_sigma)
                    z_plus, z_minus = pair_output.chunk(2, dim=0)
                elif execution == "sequential":
                    z_plus = model.tweedie(forward_plus, sigma_batch)
                    z_minus = model.tweedie(forward_minus, sigma_batch)
                else:
                    raise ValueError(
                        "antithetic_execution must be 'sequential' or 'stacked'."
                    )

                z = 0.5 * (z_plus + z_minus)
                with torch.no_grad():
                    gap = (z_plus - z_minus).detach().flatten(1)
                    d = max(1, gap.shape[1])
                    gap_norm = float(gap.norm(dim=1).mean().detach() / math.sqrt(d))
                    sigma_mean = float(sigma_batch.mean().detach())
                    rho_mean = float(rho_batch.mean().detach())
                    self._last_antithetic_stats = {
                        "active": 1.0,
                        "sigma_mean": sigma_mean,
                        "pair_scale_mean": rho_mean,
                        "pair_scale_over_sigma": rho_mean / max(sigma_mean, 1e-12),
                        "output_pair_gap": gap_norm,
                        "output_pair_gap_over_sigma": gap_norm / max(sigma_mean, 1e-12),
                    }
                return z

            self._last_antithetic_stats = {
                "active": 0.0,
                "sigma_mean": float(sigma_batch.mean().detach()),
                "pair_scale_mean": 0.0,
                "pair_scale_over_sigma": 0.0,
                "output_pair_gap": 0.0,
                "output_pair_gap_over_sigma": 0.0,
            }

            ac_noise = bool(getattr(denoise_config, "ac_noise", True))
'''
    if old_denoise_anchor not in source:
        raise RuntimeError("Could not find the expected denoising anchor. The branch may have changed.")
    source = source.replace(old_denoise_anchor, new_denoise_anchor, 1)

    old_schedule_anchor = '''        sigma_schedule, tau_schedule, rho_schedule, theorem1_tail_mask = self._build_sigma_tau_rho_schedule(K)
        sigma_dual_schedule = self._build_sigma_dual_schedule(sigma_schedule, theorem1_tail_mask)
'''
    new_schedule_anchor = '''        sigma_schedule, tau_schedule, rho_schedule, theorem1_tail_mask = self._build_sigma_tau_rho_schedule(K)
        sigma_dual_schedule = self._build_sigma_dual_schedule(sigma_schedule, theorem1_tail_mask)

        anti_cfg = getattr(self.admm_config.denoise, "antithetic", None)
        antithetic_enabled = bool(getattr(anti_cfg, "enabled", False)) if anti_cfg is not None else False
        antithetic_scope = str(getattr(anti_cfg, "scope", "theorem1_tail")).lower() if anti_cfg is not None else "theorem1_tail"
        antithetic_last_n = int(getattr(anti_cfg, "last_n", 0)) if anti_cfg is not None else 0
        antithetic_noise_source = str(getattr(anti_cfg, "noise_source", "sigma")).lower() if anti_cfg is not None else "sigma"
        antithetic_execution = str(getattr(anti_cfg, "execution", "sequential")).lower() if anti_cfg is not None else "sequential"

        if antithetic_scope not in {"full", "last_n", "theorem1_tail"}:
            raise ValueError("antithetic.scope must be 'full', 'last_n', or 'theorem1_tail'.")
        if antithetic_noise_source not in {"sigma", "rho"}:
            raise ValueError("antithetic.noise_source must be 'sigma' or 'rho'.")
        if antithetic_execution not in {"sequential", "stacked"}:
            raise ValueError("antithetic.execution must be 'sequential' or 'stacked'.")
        if antithetic_enabled:
            print(
                "[PDHG] antithetic denoising active: "
                f"scope={antithetic_scope}, last_n={antithetic_last_n}, "
                f"noise_source={antithetic_noise_source}, execution={antithetic_execution}"
            )
'''
    if old_schedule_anchor not in source:
        raise RuntimeError("Could not find the expected schedule anchor. The branch may have changed.")
    source = source.replace(old_schedule_anchor, new_schedule_anchor, 1)

    old_step_anchor = '''            theorem1_tail_active = bool(theorem1_tail_mask[step])
            theta = 0.0 if self.force_theta_zero else float(self.theta_schedule[min(step, len(self.theta_schedule) - 1)])
'''
    new_step_anchor = '''            theorem1_tail_active = bool(theorem1_tail_mask[step])
            if not antithetic_enabled:
                antithetic_active = False
            elif antithetic_scope == "full":
                antithetic_active = True
            elif antithetic_scope == "last_n":
                antithetic_active = step >= max(0, K - max(0, antithetic_last_n))
            else:
                antithetic_active = theorem1_tail_active
            antithetic_pair_scale = sigma_d if antithetic_noise_source == "sigma" else rho_d
            theta = 0.0 if self.force_theta_zero else float(self.theta_schedule[min(step, len(self.theta_schedule) - 1)])
'''
    if old_step_anchor not in source:
        raise RuntimeError("Could not find the expected iteration anchor. The branch may have changed.")
    source = source.replace(old_step_anchor, new_step_anchor, 1)

    old_call_anchor = '''                sigma=sigma_d,
                rho=rho_d,
                prior_use_type=self.admm_config.denoise.type,
                wandb=wandb
            )
'''
    new_call_anchor = '''                sigma=sigma_d,
                rho=(antithetic_pair_scale if antithetic_active else rho_d),
                prior_use_type=self.admm_config.denoise.type,
                wandb=wandb,
                antithetic=antithetic_active,
                antithetic_execution=antithetic_execution,
            )
'''
    if old_call_anchor not in source:
        raise RuntimeError("Could not find the expected main denoising call. The branch may have changed.")
    source = source.replace(old_call_anchor, new_call_anchor, 1)

    old_metric_anchor = '''                self._metric_history_add("dual_inject_norm", float(dual_inject_norm))
                self._metric_history_add("dual_inject_over_sigma", float(dual_inject_over_sigma))
'''
    new_metric_anchor = '''                self._metric_history_add("dual_inject_norm", float(dual_inject_norm))
                self._metric_history_add("dual_inject_over_sigma", float(dual_inject_over_sigma))
                antithetic_stats = getattr(self, "_last_antithetic_stats", None)
                if antithetic_stats is not None:
                    for key, value in antithetic_stats.items():
                        self._metric_history_add(f"antithetic_{key}", float(value))
'''
    if old_metric_anchor not in source:
        raise RuntimeError("Could not find the expected metric-history anchor. The branch may have changed.")
    source = source.replace(old_metric_anchor, new_metric_anchor, 1)

    pdhg_path.write_text(source, encoding="utf-8")
    py_compile.compile(pdhg_path.as_posix(), doraise=True)
    print(f"Applied and syntax-checked antithetic patch: {pdhg_path}")


In [ ]:
#@title Install Colab Dependencies
import os
import subprocess

os.chdir(REPO_DIR)
subprocess.run([PYTHON_BIN, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
print("Installed requirements-colab.txt")


In [ ]:
#@title Download The FFHQ Checkpoint If Needed
import os
import subprocess
from pathlib import Path

os.chdir(REPO_DIR)
ckpt_path = Path("pretrained-models/ffhq_10m.pt")
ckpt_path.parent.mkdir(parents=True, exist_ok=True)

if ckpt_path.exists():
    print(f"Checkpoint already present: {ckpt_path}")
else:
    subprocess.run(
        [
            "gdown",
            "--id",
            "1BGwhRWUoguF-D8wlZ65tf227gp3cDUDh",
            "-O",
            ckpt_path.as_posix(),
        ],
        check=True,
    )
    print(f"Downloaded checkpoint to: {ckpt_path}")


In [ ]:
#@title Build Single-Run Command
import json
import os
import shlex
import time
from pathlib import Path

os.chdir(REPO_DIR)
repo_dir = Path(REPO_DIR)
drive_data_dir = Path(DRIVE_FFHQ_DATA_DIR)
if not drive_data_dir.exists():
    raise FileNotFoundError(f"FFHQ dataset path not found: {drive_data_dir}")

if SAMPLER_CONFIG != "edm_pdhg" and ANTITHETIC_ENABLED:
    raise ValueError("This notebook implements antithetic denoising only for sampler=edm_pdhg.")
if ANTITHETIC_ENABLED and DENOISE_FINAL_STEP != "tweedie":
    raise ValueError("Antithetic mode currently requires DENOISE_FINAL_STEP='tweedie'.")
if ANTITHETIC_ENABLED and ANTITHETIC_SCOPE == "theorem1_tail" and int(MAP_TAIL_STEPS) <= 0:
    raise ValueError("ANTITHETIC_SCOPE='theorem1_tail' requires MAP_TAIL_STEPS > 0.")

def parse_list(text: str):
    items = []
    for raw_item in text.replace("\n", ";").split(";"):
        raw_item = raw_item.strip()
        if raw_item:
            items.append(raw_item)
    return items

metric_list = parse_list(EVAL_METRICS)
if not metric_list:
    raise ValueError("EVAL_METRICS must contain at least one metric name.")

extra_overrides = parse_list(EXTRA_HYDRA_OVERRIDES)
effective_early_stop_iter = MAX_ITER if int(EARLY_STOP_ITER) <= 0 else min(int(EARLY_STOP_ITER), int(MAX_ITER))
session_tag = SESSION_TAG.strip() or time.strftime("%Y%m%d-%H%M%S")
sampler_tag = SAMPLER_CONFIG.replace("edm_", "")
method_tag = f"anti-{ANTITHETIC_SCOPE}" if ANTITHETIC_ENABLED else "standard"
run_tag = f"{sampler_tag}_{method_tag}_{INVERSE_TASK}_{session_tag}"
run_name = f"{RUN_NAME}_{session_tag}"
save_root = repo_dir / "results" / "single_runs" / run_tag
hydra_root = repo_dir / "outputs" / "single_runs" / run_tag
run_aux_root = repo_dir / "single_runs"
latest_log_path = run_aux_root / f"{run_tag}.log"
latest_pid_path = run_aux_root / f"{run_tag}.pid"
data_end_idx = DATA_START_IDX + TOTAL_IMAGES
eval_fn_override = f"eval_fn_list=[{','.join(metric_list)}]"

run_cmd = [
    PYTHON_BIN,
    "recover_inverse2.py",
    "--config-name",
    CONFIG_NAME,
    f"sampler={SAMPLER_CONFIG}",
    f"inverse_task={INVERSE_TASK}",
    f"name={run_name}",
    f"seed={SEED}",
    "gpu=0",
    "wandb=false",
    "show_config=false",
    f"save_samples={'true' if SAVE_SAMPLES else 'false'}",
    f"save_traj={'true' if SAVE_TRAJ else 'false'}",
    f"save_traj_raw_data={'true' if SAVE_TRAJ_RAW_DATA else 'false'}",
    f"total_images={TOTAL_IMAGES}",
    f"batch_size={BATCH_SIZE}",
    "num_runs=1",
    f"inverse_task.operator.sigma={MEASUREMENT_SIGMA}",
    f"sampler.annealing_scheduler_config.num_steps={NUM_STEPS}",
    f"sampler.annealing_scheduler_config.sigma_max={SIGMA_MAX}",
    f"sampler.annealing_scheduler_config.sigma_min={SIGMA_MIN}",
    f"inverse_task.admm_config.max_iter={MAX_ITER}",
    f"++inverse_task.admm_config.early_stop={effective_early_stop_iter}",
    f"inverse_task.admm_config.denoise.final_step={DENOISE_FINAL_STEP}",
    f"++inverse_task.admm_config.denoise.ac_noise={'true' if DENOISER_AC_NOISE else 'false'}",
    f"inverse_task.admm_config.dys.gamma={DYS_GAMMA}",
    "inverse_task.admm_config.dys.lambda_schedule.activate=true",
    f"inverse_task.admm_config.dys.lambda_schedule.start={LAMBDA_START}",
    f"inverse_task.admm_config.dys.lambda_schedule.end={LAMBDA_END}",
    f"inverse_task.admm_config.dys.lambda_schedule.warmup={LAMBDA_WARMUP}",
    eval_fn_override,
    f"data.image_root_path={drive_data_dir.as_posix()}",
    f"data.start_idx={DATA_START_IDX}",
    f"data.end_idx={data_end_idx}",
]

if SAMPLER_CONFIG == "edm_pdhg":
    run_cmd.extend([
        f"++inverse_task.admm_config.pdhg.tau={TAU}",
        f"++inverse_task.admm_config.pdhg.sigma_dual={SIGMA_DUAL}",
        f"++inverse_task.admm_config.pdhg.sigma_dual_schedule_mode={SIGMA_DUAL_SCHEDULE_MODE}",
        f"++inverse_task.admm_config.pdhg.sigma_dual_schedule_scope={SIGMA_DUAL_SCHEDULE_SCOPE}",
        f"++inverse_task.admm_config.pdhg.sigma_dual_schedule_power={SIGMA_DUAL_SCHEDULE_POWER}",
        "inverse_task.admm_config.denoise.lgvd.num_steps=0",
        f"++sampler.annealing_scheduler_config.theorem1_tail_steps={int(MAP_TAIL_STEPS)}",
        f"++sampler.annealing_scheduler_config.theorem1_tail_sigma_mode={MAP_TAIL_SIGMA_MODE}",
        f"++sampler.annealing_scheduler_config.theorem1_tail_rho_power={MAP_TAIL_RHO_POWER}",
        f"++sampler.annealing_scheduler_config.theorem1_tail_rho_scale={MAP_TAIL_RHO_SCALE}",
        f"++inverse_task.admm_config.denoise.antithetic.enabled={'true' if ANTITHETIC_ENABLED else 'false'}",
        f"++inverse_task.admm_config.denoise.antithetic.scope={ANTITHETIC_SCOPE}",
        f"++inverse_task.admm_config.denoise.antithetic.last_n={int(ANTITHETIC_LAST_N)}",
        f"++inverse_task.admm_config.denoise.antithetic.noise_source={ANTITHETIC_NOISE_SOURCE}",
        f"++inverse_task.admm_config.denoise.antithetic.execution={ANTITHETIC_EXECUTION}",
    ])
    if float(MAP_TAIL_LAMBDA) > 0.0:
        run_cmd.append(
            f"++sampler.annealing_scheduler_config.theorem1_tail_lambda={float(MAP_TAIL_LAMBDA)}"
        )
else:
    run_cmd.extend([
        f"inverse_task.admm_config.rho={RHO}",
        f"inverse_task.admm_config.ml.lr={ADMM_ML_LR}",
        f"inverse_task.admm_config.denoise.lgvd.num_steps={ADMM_LGVD_NUM_STEPS}",
    ])

run_cmd.extend(extra_overrides)
run_cmd.extend([
    f"save_dir={save_root.as_posix()}",
    f"hydra.run.dir={hydra_root.as_posix()}",
])

if ANTITHETIC_ENABLED:
    if ANTITHETIC_SCOPE == "full":
        planned_antithetic_steps = effective_early_stop_iter
    elif ANTITHETIC_SCOPE == "last_n":
        planned_antithetic_steps = min(effective_early_stop_iter, max(0, int(ANTITHETIC_LAST_N)))
    else:
        planned_antithetic_steps = min(effective_early_stop_iter, max(0, int(MAP_TAIL_STEPS)))
else:
    planned_antithetic_steps = 0
planned_sample_equivalent_nfe = effective_early_stop_iter + planned_antithetic_steps

print(f"Run tag: {run_tag}")
print(f"Sampler: {SAMPLER_CONFIG}")
print(f"Save root: {save_root}")
print(f"Dataset slice: [{DATA_START_IDX}, {data_end_idx})")
print(f"max_iter: {MAX_ITER}")
print(f"early_stop_iter: {effective_early_stop_iter}")
if SAMPLER_CONFIG == "edm_pdhg":
    print(f"tau: {TAU}")
    print(f"sigma_dual: {SIGMA_DUAL}")
    print(
        f"sigma_dual schedule: mode={SIGMA_DUAL_SCHEDULE_MODE}, "
        f"scope={SIGMA_DUAL_SCHEDULE_SCOPE}, power={SIGMA_DUAL_SCHEDULE_POWER}"
    )
    print("PDHG LGVD steps: 0")
    print(f"MAP-style tail steps: {MAP_TAIL_STEPS}")
    print(
        "MAP-tail lambda: "
        + (str(MAP_TAIL_LAMBDA) if float(MAP_TAIL_LAMBDA) > 0.0 else "inferred as sigma_switch^2 / TAU")
    )
    print(f"Antithetic enabled: {ANTITHETIC_ENABLED}")
    if ANTITHETIC_ENABLED:
        print(f"Antithetic scope: {ANTITHETIC_SCOPE}")
        print(f"Pair scale source: {ANTITHETIC_NOISE_SOURCE}")
        print(f"Pair execution: {ANTITHETIC_EXECUTION}")
        print(f"Planned paired iterations: {planned_antithetic_steps}")
        print(
            "Planned sample-equivalent NFE: "
            f"{planned_sample_equivalent_nfe} "
            "(a paired iteration performs two denoiser evaluations, even when stacked into one batched call)"
        )
else:
    print(f"rho: {RHO}")
    print(f"ADMM ml.lr: {ADMM_ML_LR}")
    print(f"ADMM LGVD steps: {ADMM_LGVD_NUM_STEPS}")
print(f"Denoiser AC re-noise outside antithetic iterations: {DENOISER_AC_NOISE}")
print(f"Metrics: {metric_list}")
if extra_overrides:
    print(f"Extra overrides: {extra_overrides}")

print("\nCommand:\n")
print(" ".join(shlex.quote(part) for part in run_cmd))

last_context = {
    "run_tag": run_tag,
    "run_name": run_name,
    "save_root": save_root.as_posix(),
    "hydra_root": hydra_root.as_posix(),
    "latest_log_path": latest_log_path.as_posix(),
    "latest_pid_path": latest_pid_path.as_posix(),
    "run_cmd": run_cmd,
    "antithetic_enabled": bool(ANTITHETIC_ENABLED),
    "antithetic_scope": str(ANTITHETIC_SCOPE),
    "planned_antithetic_steps": int(planned_antithetic_steps),
    "planned_sample_equivalent_nfe": int(planned_sample_equivalent_nfe),
}

run_aux_root.mkdir(parents=True, exist_ok=True)
context_path = run_aux_root / f"{run_tag}.context.json"
with context_path.open("w", encoding="utf-8") as handle:
    json.dump(last_context, handle, indent=2)
print(f"Context saved to: {context_path}")


## What the implementation does

For an active antithetic iteration, the notebook evaluates

\[
D_{\sigma_k}(v^{k+1}+a_kn^k),\qquad D_{\sigma_k}(v^{k+1}-a_kn^k)
\]

with the **same** Gaussian draw and averages the outputs. The default `ANTITHETIC_NOISE_SOURCE="sigma"` sets \(a_k=\sigma_k\), so each marginal input has the score model's training corruption scale. `sequential` execution is memory-safe; `stacked` concatenates the pair along the batch dimension and may require reducing `BATCH_SIZE`.

For a fair compute comparison, one paired iteration counts as **two sample-equivalent score evaluations**, even when the two images are sent through a single stacked batch.

The direct theory applies to `final_step="tweedie"` and zero inner LGVD steps. A multi-step stochastic inner denoising trajectory would need every inner noise draw to be coupled antithetically as well.


In [ ]:
#@title Launch The Run In Background
import os
import subprocess
from pathlib import Path

os.chdir(REPO_DIR)
latest_log_path = Path(last_context["latest_log_path"])
latest_pid_path = Path(last_context["latest_pid_path"])
latest_log_path.parent.mkdir(parents=True, exist_ok=True)

with latest_log_path.open("w", encoding="utf-8") as log_handle:
    process = subprocess.Popen(
        last_context["run_cmd"],
        cwd=REPO_DIR,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        text=True,
    )

latest_pid_path.write_text(str(process.pid), encoding="utf-8")
print(f"PID: {process.pid}")
print(f"Log: {latest_log_path}")
print(f"Save root: {last_context['save_root']}")


In [ ]:
#@title Show Recent Log Lines
from pathlib import Path

log_path = Path(last_context["latest_log_path"])
if not log_path.exists():
    raise FileNotFoundError(f"Log file not found: {log_path}")

lines = log_path.read_text(encoding="utf-8", errors="ignore").splitlines()
tail = lines[-int(LOG_TAIL_LINES):]
print("\n".join(tail) if tail else "<log is empty>")


In [ ]:
#@title Show Results And Artifacts
import json
from pathlib import Path

save_root = Path(last_context["save_root"])
metrics_matches = sorted(save_root.rglob("metrics.json"))
metric_history_matches = sorted(save_root.rglob("metric_history.json"))
eval_matches = sorted(save_root.rglob("eval.md"))
grid_matches = sorted(save_root.rglob("grid_results.png"))

print("Artifacts:\n")
print(f"save_root: {save_root}")
print(f"metrics.json matches: {[p.as_posix() for p in metrics_matches]}")
print(f"metric_history.json matches: {[p.as_posix() for p in metric_history_matches]}")
print(f"eval.md matches: {[p.as_posix() for p in eval_matches]}")
print(f"grid_results.png matches: {[p.as_posix() for p in grid_matches]}")

if eval_matches:
    eval_path = eval_matches[0]
    print("\neval.md:\n")
    print(eval_path.read_text(encoding="utf-8", errors="ignore"))
else:
    print("\nNo eval.md found yet.")

if metrics_matches:
    metrics_path = metrics_matches[0]
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    print("\nmetrics.json:\n")
    print(json.dumps(metrics, indent=2))
else:
    print("\nNo metrics.json found yet.")

if metric_history_matches:
    metric_history_path = metric_history_matches[0]
    metric_history = json.loads(metric_history_path.read_text(encoding="utf-8"))
    print("\nmetric_history.json summary:\n")
    print(f"path: {metric_history_path}")
    if isinstance(metric_history, dict) and "runs" in metric_history:
        print(f"runs captured: {len(metric_history['runs'])}")
        history_view = metric_history["runs"][0] if metric_history["runs"] else {}
    else:
        history_view = metric_history
    series_keys = [key for key, value in history_view.items() if isinstance(value, list)]
    print(f"series keys: {series_keys}")
    for key in series_keys:
        series = history_view[key]
        if not series:
            print(f"{key}: <empty>")
            continue
        tail = series[-5:] if len(series) > 5 else series
        print(f"{key}: len={len(series)} final={series[-1]} tail={tail}")
else:
    print("\nNo metric_history.json found yet.")


print("\nRun-method context:\n")
for key in [
    "antithetic_enabled",
    "antithetic_scope",
    "planned_antithetic_steps",
    "planned_sample_equivalent_nfe",
]:
    if key in last_context:
        print(f"{key}: {last_context[key]}")


In [ ]:
#@title Copy Run Artifacts To Drive
import shutil
from pathlib import Path

export_root = Path(DRIVE_EXPORT_DIR)
export_root.mkdir(parents=True, exist_ok=True)

targets = [
    Path(last_context["save_root"]),
    Path(last_context["hydra_root"]),
    Path(last_context["latest_log_path"]),
]

for src in targets:
    if not src.exists():
        print(f"Skipping missing path: {src}")
        continue

    dst = export_root / src.name
    if src.is_dir():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    print(f"Copied {src} -> {dst}")
